# Прогноз дохода — WMAE. 

In [ ]:
import pandas as pd, numpy as np, glob, os, warnings
import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import StratifiedKFold
from scipy.optimize import minimize
warnings.filterwarnings('ignore'); np.random.seed(42)

def wmae(y_true, y_pred, w):  
    y_true=np.asarray(y_true,float); y_pred=np.asarray(y_pred,float); w=np.asarray(w,float)
    return float(np.average(np.abs(y_true-y_pred), weights=w))
print('libs OK | lgb', lgb.__version__, '| xgb', xgb.__version__)

In [ ]:

KNOWN_CATEGORICAL=['gender','adminarea','city_smart_name','dp_ewb_last_employment_position','addrref','period_last_act_ad']
KEEP_AS_STRING=set(KNOWN_CATEGORICAL)|{'dt'}; NA_TOKENS=['','nan','NaN','None','NULL','NA','<NA>']
def find_file(patterns):
    dirs=['.','..','../..','solution/data','../data', r'C:\Users\korab\OneDrive\Рабочий стол\ФинТех\Хахатон']
    c=[]
    for d in dirs:
        for p in patterns: c+=glob.glob(os.path.join(d,p))
    c=[x for x in c if os.path.isfile(x)]; ne=[x for x in c if 'example' not in os.path.basename(x).lower()]
    return sorted(set(ne or c), key=os.path.getsize, reverse=True)[0]
def coerce_numeric(s):
    s=(s.astype('string').str.strip().str.replace(' ','',regex=False).str.replace(' ','',regex=False).str.replace(',','.',regex=False))
    return pd.to_numeric(s, errors='coerce')
def load_table(path):
    with open(path,'rb') as f: is_xlsx=f.read(2)==b'PK'
    df=pd.read_excel(path) if is_xlsx else pd.read_csv(path,sep=';',dtype='string',keep_default_na=True,na_values=NA_TOKENS)
    df.columns=[str(c).strip() for c in df.columns]; cats=[]
    for c in df.columns:
        if c in KEEP_AS_STRING:
            df[c]=df[c].astype('string').str.strip()
            if c!='dt': cats.append(c)
            continue
        conv=coerce_numeric(df[c]); nn=int(df[c].notna().sum())
        if nn==0 or conv.notna().sum()/nn>=0.8: df[c]=conv
        else: df[c]=df[c].astype('string').str.strip(); cats.append(c)
    df['id']=pd.to_numeric(df['id'],errors='coerce').astype('Int64')
    return df,cats
train,_=load_table(find_file(['*train*.csv','*train*']))
test,_ =load_table(find_file(['*test*.csv','*test*']))
sample=pd.read_csv(find_file(['*sample*submission*','sample_submission*']),sep=';')
sample.columns=[c.strip() for c in sample.columns]; sample['id']=pd.to_numeric(sample['id'],errors='coerce').astype('Int64')
print('train',train.shape,'| test',test.shape,'(кредитные признаки — числовые)')

In [ ]:
# === Препроцессинг: даты, n_missing, частотное кодирование, категории, дроп констант ===
CAT_COLS=['gender','adminarea','city_smart_name','addrref','dp_ewb_last_employment_position','dp_ewb_last_organization']
def to_dt(s): return pd.to_datetime(s.astype('string'), errors='coerce')
def add_dates(df):
    df=df.copy(); dt=to_dt(df['dt']) if 'dt' in df else pd.Series(pd.NaT,index=df.index)
    if 'period_last_act_ad' in df:
        p=to_dt(df['period_last_act_ad'])
        df['plad_year']=p.dt.year; df['plad_month']=p.dt.month
        df['plad_months_ago']=(dt.dt.year-p.dt.year)*12+(dt.dt.month-p.dt.month); df['plad_is_present']=p.notna().astype('int8')
    return df
raw_feats=[c for c in train.columns if c not in ('id','target','w','dt','period_last_act_ad')]
train=add_dates(train); test=add_dates(test)
train['n_missing']=train[raw_feats].isna().sum(axis=1).astype('int32')
test['n_missing']=test[[c for c in raw_feats if c in test]].isna().sum(axis=1).astype('int32')
for c in CAT_COLS:
    if c in train:
        fr=pd.concat([train[c].astype('string'),test[c].astype('string')],ignore_index=True).value_counts(normalize=True)
        train[c+'_freq']=train[c].astype('string').map(fr).astype('float32').fillna(0)
        test[c+'_freq']=test[c].astype('string').map(fr).astype('float32').fillna(0)
for c in CAT_COLS:
    if c in train:
        train[c]=train[c].astype('string').fillna('__nan__').astype('category')
        test[c]=test[c].astype('string').fillna('__nan__').astype('category')
train=train.drop(columns=['dt','period_last_act_ad'],errors='ignore'); test=test.drop(columns=['dt','period_last_act_ad'],errors='ignore')
nun=train.nunique(dropna=True); const=[c for c in nun[nun<=1].index if c not in (['id','target','w']+CAT_COLS)]
train=train.drop(columns=const); test=test.drop(columns=[c for c in const if c in test])
y=train['target'].astype('float64').values; w=train['w'].astype('float64').values; id_test=test['id'].values
feat=[c for c in train.columns if c not in ('id','target','w') and c in test.columns]
final_cat=[c for c in CAT_COLS if c in feat]
X=train[feat].copy(); Xte=test[feat].copy()
for c in final_cat:
    u=pd.api.types.union_categoricals([X[c],Xte[c]],ignore_order=True).categories
    X[c]=pd.Categorical(X[c],categories=u); Xte[c]=pd.Categorical(Xte[c],categories=u)
print('признаков:',len(feat),'| категорий:',len(final_cat))

In [ ]:
# === Feature engineering===
INCOME_PROXY=['dp_ils_avg_salary_1y','dp_ils_avg_salary_2y','salary_6to12m_avg','first_salary_income','incomeValue','salary_median_in_gex_r1','dp_ils_paymentssum_avg_12m']
def safe_ratio(a,b): r=a.astype('float64')/b.astype('float64'); return r.replace([np.inf,-np.inf],np.nan)
def add_features(df):
    df=df.copy(); cols=set(df.columns); has=lambda c: c in cols; added=[]
    proxy=[c for c in INCOME_PROXY if has(c)]
    if len(proxy)>=2:
        P=df[proxy].astype('float64')
        df['income_proxy_max']=P.max(1); df['income_proxy_mean']=P.mean(1); df['income_proxy_min']=P.min(1)
        df['income_proxy_std']=P.std(1); df['income_proxy_n']=P.notna().sum(1).astype('int16')
    if has('dp_ils_avg_salary_1y') and has('dp_ils_avg_salary_2y'):
        df['salary_trend_1y_2y']=safe_ratio(df['dp_ils_avg_salary_1y'],df['dp_ils_avg_salary_2y'])
    for cr,db,nm in [('turn_cur_cr_avg_v2','turn_cur_db_avg_v2','turn_cur_cr_db_ratio'),('avg_cur_cr_turn','avg_cur_db_turn','avg_cur_cr_db_ratio')]:
        if has(cr) and has(db):
            df[nm]=safe_ratio(df[cr],df[db]); df[nm.replace('ratio','sum')]=df[cr].astype('float64')+df[db].astype('float64')
    if has('dp_ils_avg_salary_1y') and has('turn_cur_cr_avg_v2'):
        df['salary_vs_turn_cr']=safe_ratio(df['dp_ils_avg_salary_1y'],df['turn_cur_cr_avg_v2'])
    sp=[c for c in df.columns if 'by_category__amount' in c]
    if sp:
        S=df[sp].astype('float64'); df['spend_cat_sum']=S.sum(1,min_count=1); df['spend_cat_n']=S.notna().sum(1).astype('int16'); df['spend_cat_max']=S.max(1)
    if has('hdb_bki_total_max_overdue_sum') and has('hdb_bki_total_max_limit'):
        df['overdue_to_limit']=safe_ratio(df['hdb_bki_total_max_overdue_sum'],df['hdb_bki_total_max_limit'])
    ov=[c for c in ['hdb_ovrd_sum','ovrd_sum'] if has(c)]
    if ov: df['ovrd_total']=df[ov].astype('float64').sum(1,min_count=1)
    for g,tok in [('bki','bki'),('spend','by_category'),('ils','dp_ils')]:
        gc=[c for c in df.columns if tok in c]
        if len(gc)>=3: df[f'n_missing_{g}']=df[gc].isna().sum(1).astype('int16')
    return df
X=add_features(X); Xte=add_features(Xte)
common=[c for c in X.columns if c in Xte.columns]; X,Xte=X[common],Xte[common]
# дроп признаков
allnan=[c for c in Xte.columns if Xte[c].isna().all() and c not in final_cat]
X=X.drop(columns=allnan); Xte=Xte.drop(columns=allnan)
cat=[c for c in final_cat if c in X.columns]; y_min,y_max=float(y.min()),float(y.max())
print('итог признаков:',X.shape[1],'| дропнуто пустых в test:',allnan)

In [ ]:
# === Варианты матрицы под каждую модель ===
def as_codes(df):
    df=df.copy()
    for c in cat: df[c]=df[c].cat.codes.astype('int32')
    return df
def as_str(df):
    df=df.copy()
    for c in cat: df[c]=df[c].astype('string').fillna('__nan__').astype(str)
    return df
Xc,Xtec=as_codes(X),as_codes(Xte)   # XGBoost — ordinal-коды
Xs,Xtes=as_str(X),as_str(Xte)       # CatBoost — строки
bins=pd.qcut(y,10,duplicates='drop'); bins=pd.Series(bins).astype('category').cat.codes.values
folds=list(StratifiedKFold(5,shuffle=True,random_state=42).split(np.zeros(len(y)),bins))
oofs={}; tests={}
print('фолдов:',len(folds))

In [ ]:
# === LightGBM ===
LGB=dict(objective='regression_l1',n_estimators=3500,learning_rate=0.03,num_leaves=96,min_child_samples=50,
         subsample=0.85,subsample_freq=1,colsample_bytree=0.7,reg_lambda=3.0,reg_alpha=0.5,random_state=42,n_jobs=-1,verbose=-1)
for log_t,nm in ((False,'lgb_raw'),(True,'lgb_log')):
    oof=np.zeros(len(y)); tp=np.zeros(len(Xte))
    for tr,va in folds:
        yf=np.log1p(y[tr]) if log_t else y[tr]
        m=lgb.LGBMRegressor(**LGB)
        m.fit(X.iloc[tr],yf,sample_weight=w[tr],
              eval_set=[(X.iloc[va],np.log1p(y[va]) if log_t else y[va])],eval_sample_weight=[w[va]],
              eval_metric='l1',categorical_feature=cat,
              callbacks=[lgb.early_stopping(150,verbose=False),lgb.log_evaluation(0)])
        pv=m.predict(X.iloc[va]); ptt=m.predict(Xte)
        if log_t: pv,ptt=np.expm1(pv),np.expm1(ptt)
        oof[va]=np.clip(pv,y_min,y_max); tp+=np.clip(ptt,y_min,y_max)/len(folds)
    oofs[nm]=oof; tests[nm]=tp; print(f'{nm}: OOF WMAE = {wmae(y,oof,w):,.0f}')

In [ ]:
# === XGBoost ===
XGB=dict(objective='reg:absoluteerror',tree_method='hist',n_estimators=2000,learning_rate=0.03,max_depth=9,
         min_child_weight=25,subsample=0.9,colsample_bytree=0.75,reg_alpha=4.0,reg_lambda=3.0,
         random_state=42,n_jobs=-1,eval_metric='mae',early_stopping_rounds=120,verbosity=0)
oof=np.zeros(len(y)); tp=np.zeros(len(Xte))
for tr,va in folds:
    m=xgb.XGBRegressor(**XGB)
    m.fit(Xc.iloc[tr],y[tr],sample_weight=w[tr],eval_set=[(Xc.iloc[va],y[va])],sample_weight_eval_set=[w[va]],verbose=False)
    oof[va]=np.clip(m.predict(Xc.iloc[va]),y_min,y_max); tp+=np.clip(m.predict(Xtec),y_min,y_max)/len(folds)
oofs['xgb_raw']=oof; tests['xgb_raw']=tp; print(f'xgb_raw: OOF WMAE = {wmae(y,oof,w):,.0f}')

In [ ]:
# === CatBoost (MAE)  ===
CB=dict(loss_function='MAE',eval_metric='MAE',iterations=2000,learning_rate=0.04,depth=8,l2_leaf_reg=3.0,
        random_seed=42,od_type='Iter',od_wait=120,boosting_type='Plain',max_ctr_complexity=1,
        leaf_estimation_iterations=1,bootstrap_type='Bernoulli',subsample=0.85,thread_count=-1,verbose=0,allow_writing_files=False)
oof=np.zeros(len(y)); tp=np.zeros(len(Xte))
for tr,va in folds:
    trp=Pool(Xs.iloc[tr],y[tr],weight=w[tr],cat_features=cat); vap=Pool(Xs.iloc[va],y[va],weight=w[va],cat_features=cat)
    m=CatBoostRegressor(**CB); m.fit(trp,eval_set=vap,use_best_model=True)
    oof[va]=np.clip(m.predict(Xs.iloc[va]),y_min,y_max); tp+=np.clip(m.predict(Xtes),y_min,y_max)/len(folds)
oofs['cat_raw']=oof; tests['cat_raw']=tp; print(f'cat_raw: OOF WMAE = {wmae(y,oof,w):,.0f}')

In [ ]:
# === Бленд по OOF ===
names=list(oofs.keys()); P=np.vstack([oofs[n] for n in names]).T
def loss(a):
    a=np.clip(a,0,None); s=a.sum(); return 1e18 if s<=0 else wmae(y,P@(a/s),w)
best=None
for init in [np.ones(len(names))/len(names)]+list(np.eye(len(names))):
    r=minimize(loss,init,method='Nelder-Mead',options=dict(maxiter=4000,xatol=1e-4,fatol=1e-3))
    if best is None or r.fun<best.fun: best=r
W=np.clip(best.x,0,None); W=W/W.sum()
print('OOF-веса бленда:', {n:round(float(a),3) for n,a in zip(names,W)})
te_bl=sum(a*tests[n] for a,n in zip(W,names)); oof_bl=P@W
print(f'бленд OOF WMAE = {wmae(y,oof_bl,w):,.0f}')



ALPHA, T = 1.06, 90000.0
pred=te_bl.copy(); pred[pred>T]=pred[pred>T]*ALPHA
pred=np.clip(pred, y_min, y_max)

sub=sample[['id']].merge(pd.DataFrame({'id':pd.array(id_test,dtype='Int64'),'predict':pred}),on='id',how='left')
sub['predict']=np.clip(sub['predict'].astype('float64'), y_min, y_max)
assert np.isfinite(sub['predict']).all() and (sub['id'].values==sample['id'].values).all()
sub.to_csv('submit_v7_stack_a106.csv', sep=';', decimal=',', index=False)      
sub.to_csv('submit_v7_stack_a106_dot.csv', sep=';', decimal='.', index=False)  
print(f'ГОТОВО: submit_v7_stack_a106.csv | строк={len(sub)} mean={sub.predict.mean():,.0f} max={sub.predict.max():,.0f}')
sub.head(10)